# Guía del estudiante — Monitoreo del servidor CEFIT

En esta guía vas a aprender qué significan los números que aparecen en el panel de Grafana.

Cada número tiene un nombre: se llama **KPI** (*Key Performance Indicator* = Indicador Clave de Rendimiento).

Piénsalo así: un médico mira la temperatura, la presión y el pulso para saber si una persona está sana.  
Nosotros miramos 9 KPIs para saber si el servidor está sano.

---

**Cómo usar esta guía:**
1. Lee cada celda de texto (como esta).
2. Ejecuta cada celda de código presionando **Shift + Enter**.
3. Observa el resultado.

## Los 9 KPIs que vamos a estudiar

| # | KPI | ¿Qué vigila? |
|---|---|---|
| 1 | Request Rate | ¿Cuántas personas entran por segundo? |
| 2 | Error Rate | ¿Qué porcentaje de visitas falla? |
| 3 | Latencia p95 | ¿Qué tan rápido responde el servidor? |
| 4 | Rate por Status | ¿Qué tipo de respuestas da el servidor? |
| 5 | p50 / p95 / p99 | ¿Cómo es la experiencia de todos los usuarios? |
| 6 | Heap Memory | ¿Cuánta memoria RAM usa el servidor? |
| 7 | CPU Usage | ¿Qué tan ocupado está el procesador? |
| 8 | Event Loop Lag | ¿El servidor está atascado con algo? |
| 9 | Active Handles | ¿Cuántas conexiones tiene abiertas? |

---
## KPI 1 — Request Rate
### ¿Cuántas peticiones llegan al servidor cada segundo?

Imagina que tienes una tienda. Quieres saber cuántos clientes entran **por minuto**.  
En un servidor, queremos saber cuántas **peticiones HTTP** llegan **por segundo**.

La fórmula es:

$$\text{Request Rate} = \frac{\text{total de peticiones}}{\text{segundos que observamos}}$$

Datos del ejercicio: en **5 minutos** (= 300 segundos) llegaron **1500 peticiones** al servidor.

In [ ]:
total_peticiones = 1500
segundos = 300
request_rate = total_peticiones / segundos
print("request_rate =", request_rate)

**¿Qué significa el número que obtuviste?**

| Valor | Estado del servidor |
|---|---|
| 0 req/s | ⚠️ Nadie está usando el servidor (¿está caído?) |
| 1 – 10 req/s | ✅ Tráfico normal |
| > 50 req/s | ⚡ Mucho tráfico — el servidor trabaja fuerte |

---
## KPI 2 — Error Rate
### ¿Qué porcentaje de las peticiones terminó con error?

No todas las peticiones al servidor terminan bien. Algunas fallan con **error 500** (error del servidor).

> 👉 Un error **400** o **404** es culpa del cliente (escribió mal la URL, no tiene permiso, etc.).  
> Un error **500** es culpa del servidor (hay un bug en el código).

La fórmula es:

$$\text{Error Rate (\%)} = \frac{\text{peticiones con error 5xx}}{\text{total de peticiones}} \times 100$$

Datos del ejercicio: de **1000 peticiones**, **12** respondieron con error 500.

In [ ]:
peticiones_con_error = 12
total_peticiones = 1000
error_rate = (peticiones_con_error / total_peticiones) * 100
print("error_rate =", error_rate, "%")

**¿Qué significa ese porcentaje?**

| Valor | Estado |
|---|---|
| 0 % | ✅ Perfecto — ningún error |
| < 1 % | ✅ Aceptable en producción |
| > 5 % | 🔴 Problema serio — hay que revisar el código |

---
## KPI 3 — Latencia p95
### ¿Qué tan rápido responde el servidor para la mayoría?

La **latencia** es el tiempo que tarda el servidor en responder.

Pero no todas las peticiones tardan lo mismo. Algunas son rápidas, otras lentas.  
Para saber si el servidor es rápido en general, usamos el **percentil 95 (p95)**.

> 💡 **¿Qué es un percentil?**  
> Si tienes 100 peticiones ordenadas de más rápida a más lenta,  
> el p95 es el tiempo de la petición número 95.  
> Es decir: el 95% de los usuarios tardó **ese tiempo o menos**.

### Paso 1 — Tiempos de respuesta ordenados

Tenemos 20 tiempos de respuesta ya ordenados de menor a mayor (en milisegundos):  
`[8, 10, 12, 15, 18, 20, 22, 25, 28, 30, 35, 40, 45, 50, 60, 70, 80, 95, 200, 950]`

In [ ]:
tiempos = [8, 10, 12, 15, 18, 20, 22, 25, 28, 30,
           35, 40, 45, 50, 60, 70, 80, 95, 200, 950]
print("total de tiempos:", len(tiempos))
print("más rápido:", tiempos[0], "ms")
print("más lento:", tiempos[19], "ms")

### Paso 2 — Encontrar la posición del p95

La fórmula para encontrar la posición del percentil 95 en una lista de **n** datos:

$$\text{posición} = \left\lceil \frac{95}{100} \times n \right\rceil$$

$\lceil \cdot \rceil$ significa **redondear hacia arriba** (si da 18.5, usamos 19).

Con n = 20: $0.95 \times 20 = 19$  

> En Python los índices empiezan en **0**, no en 1.  
> La posición 19 corresponde al **índice 18**.

In [ ]:
n = 20
posicion_p95 = 0.95 * n
print("posicion_p95 =", posicion_p95)

### Paso 3 — Leer el valor del p95

La posición 19 en una lista de Python es el **índice 18** (porque Python empieza a contar desde 0).

In [ ]:
p95 = tiempos[18]
print("p95 =", p95, "ms")

### Paso 4 — Comparar p95 con el promedio

El **promedio** suma todos los tiempos y divide entre la cantidad.  
Suma de los 20 tiempos = 8+10+12+15+18+20+22+25+28+30+35+40+45+50+60+70+80+95+200+950

In [ ]:
suma_tiempos = 8+10+12+15+18+20+22+25+28+30+35+40+45+50+60+70+80+95+200+950
n = 20
promedio = suma_tiempos / n
print("promedio =", promedio, "ms")
print("p95 =", p95, "ms")

**¿Qué significa la latencia p95?**

| Valor | Estado |
|---|---|
| < 100 ms | ✅ Rápido — el usuario no lo nota |
| 100 – 500 ms | ⚠️ Aceptable |
| > 500 ms | 🔴 Lento — el usuario lo siente |

> El **promedio engaña**. Si 19 peticiones tardan 10 ms y 1 tarda 2000 ms,  
> el promedio dice ~100 ms pero un usuario real tuvo que esperar 2 segundos.  
> El p95 lo revela.

---
## KPI 4 — Request Rate por código de respuesta
### ¿Qué tipo de respuestas da el servidor?

Cada respuesta del servidor tiene un **código HTTP** que dice si todo fue bien o mal.

| Código | Significa |
|---|---|
| 200 | ✅ Todo bien |
| 201 | ✅ Se creó algo nuevo |
| 401 | ⚠️ No tienes permiso (token inválido) |
| 404 | ⚠️ La página no existe |
| 500 | 🔴 Error del servidor |

Datos del ejercicio: en **5 minutos** el servidor respondió con 850 éxitos (200), 60 rechazos (401) y 5 errores (500).

In [ ]:
respuestas_200 = 850
respuestas_401 = 60
respuestas_500 = 5
total = respuestas_200 + respuestas_401 + respuestas_500
segundos = 300
print("rate 200 =", respuestas_200 / segundos, "req/s")
print("rate 401 =", respuestas_401 / segundos, "req/s")
print("rate 500 =", respuestas_500 / segundos, "req/s")

In [ ]:
porcentaje_401 = (respuestas_401 / total) * 100
print("porcentaje_401 =", porcentaje_401, "%")

---
## KPI 5 — Latencia: p50, p95 y p99 juntos
### ¿Cómo es la experiencia de TODOS los usuarios?

Con un solo percentil solo vemos una parte. Con tres percentiles vemos el panorama completo:

- **p50** = el usuario del medio (la mitad tardó menos que esto)
- **p95** = el usuario lento (1 de cada 20 tardó más)
- **p99** = el peor caso (1 de cada 100 tardó más)

Con la lista de 20 tiempos, los índices en Python son:

| Percentil | Cálculo | Posición | Índice en Python |
|---|---|---|---|
| p50 | 0.50 × 20 = 10 | 10 | 9 |
| p95 | 0.95 × 20 = 19 | 19 | 18 |
| p99 | 0.99 × 20 = 19.8 → redondear a 20 | 20 | 19 |

In [ ]:
tiempos = [8, 10, 12, 15, 18, 20, 22, 25, 28, 30,
           35, 40, 45, 50, 60, 70, 80, 95, 200, 950]
p50 = tiempos[9]
p95 = tiempos[18]
p99 = tiempos[19]
print("p50 =", p50, "ms")
print("p95 =", p95, "ms")
print("p99 =", p99, "ms")

In [ ]:
diferencia = p99 - p50
print("diferencia p99 - p50 =", diferencia, "ms")

**Cómo interpretar los tres percentiles juntos:**

- Si **p50 ≈ p95 ≈ p99** → todos los usuarios tienen experiencia similar ✅
- Si **p99 >> p95 >> p50** → hay usuarios con experiencia muy mala ⚠️

---
## KPI 6 — Heap Memory
### ¿Cuánta memoria RAM está usando el servidor?

El servidor necesita memoria RAM para trabajar. Esa memoria se llama **heap**.

- **Heap usada** = la memoria que está ocupada ahora mismo
- **Heap total** = la memoria total que tiene disponible

La fórmula para saber qué porcentaje está usando:

$$\text{Uso (\%)} = \frac{\text{heap usada}}{\text{heap total}} \times 100$$

Datos del ejercicio: el servidor está usando **95 MB** de los **150 MB** disponibles.

In [ ]:
heap_usada = 95
heap_total = 150
uso = (heap_usada / heap_total) * 100
print("uso heap =", uso, "%")

**¿Qué significa ese porcentaje?**

| Porcentaje | Estado |
|---|---|
| < 70 % | ✅ Normal |
| 70 – 90 % | ⚠️ Empezar a vigilar |
| > 90 % | 🔴 Peligro — el servidor puede caerse |

### ¿Qué es un memory leak?

Un **memory leak** (fuga de memoria) ocurre cuando el servidor acumula memoria y **nunca la libera**.

En Grafana se ve así: la línea de heap usada sube, sube y sube… sin bajar nunca.

```
heap usada
   │            /
   │           /
   │          /   ← sube sin parar = memory leak
   │         /
   └─────────────→ tiempo
```

En condiciones normales la línea **sube y baja** porque el servidor libera memoria periódicamente.

---
## KPI 7 — CPU Usage
### ¿Qué porcentaje del procesador está usando el servidor?

El procesador (CPU) ejecuta el código. Queremos saber qué tan ocupado está.

La fórmula es:

$$\text{CPU (\%)} = \frac{\text{segundos de CPU usados}}{\text{segundos reales}} \times 100$$

Datos del ejercicio: en **60 segundos reales**, el servidor usó **15 segundos** de CPU.

In [ ]:
segundos_cpu = 15
segundos_reales = 60
cpu = (segundos_cpu / segundos_reales) * 100
print("cpu =", cpu, "%")

**¿Qué significa ese porcentaje?**

| Valor | Estado |
|---|---|
| < 30 % | ✅ Normal |
| 30 – 70 % | ⚠️ Carga moderada |
| > 70 % durante mucho tiempo | 🔴 Problema — investigar |

---
## KPI 8 — Event Loop Lag
### ¿Está el servidor atascado procesando algo?

Node.js (el servidor) procesa las tareas **una por una**, en un bucle llamado **event loop**.

Si una tarea tarda mucho, todas las demás esperan. Ese tiempo de espera es el **lag**.

**Analogía:** es como una ventanilla de banco. Si el cajero atiende a alguien que tiene 50 trámites,  
todos los demás esperan de pie aunque su trámite sea simple.

La fórmula es:

$$\text{Lag} = T_{\text{real}} - T_{\text{esperado}}$$

Datos del ejercicio: el servidor debía responder a los **100 ms** pero respondió a los **135 ms**.

In [ ]:
momento_esperado = 100
momento_real = 135
lag = momento_real - momento_esperado
print("lag =", lag, "ms")

**¿Qué significa el lag?**

| Valor | Estado |
|---|---|
| < 10 ms | ✅ El servidor está libre |
| 10 – 100 ms | ⚠️ Hay algo que tarda — investigar |
| > 100 ms | 🔴 El servidor está atascado — los usuarios esperan |

---
## KPI 9 — Active Handles
### ¿Cuántas conexiones tiene abiertas el servidor?

Cada vez que un usuario se conecta al servidor, se abre un **handle** (una conexión).  
Cuando el usuario termina, esa conexión se cierra.

Si el número de handles sube mucho y **nunca baja**, hay una fuga de conexiones.

Caso 1: tráfico bajo con pocas conexiones — situación normal.

In [ ]:
request_rate = 5
active_handles = 8
print("request_rate =", request_rate, "req/s")
print("active_handles =", active_handles)

Caso 2: tráfico muy bajo pero con cientos de conexiones abiertas — situación sospechosa.

In [ ]:
request_rate = 2
active_handles = 950
print("request_rate =", request_rate, "req/s")
print("active_handles =", active_handles)

**¿Qué significa el número de handles?**

| Situación | Estado |
|---|---|
| Handles proporcionales al tráfico | ✅ Normal |
| Handles suben pero el tráfico no | ⚠️ Investigar |
| Handles suben sin parar | 🔴 Fuga de conexiones |

---
## Resumen final — Los 9 KPIs

| # | KPI | Fórmula | Saludable | Alerta |
|---|---|---|---|---|
| 1 | Request Rate | peticiones / segundos | > 0 req/s | = 0 (sin tráfico) |
| 2 | Error Rate | (errores 5xx / total) × 100 | < 1% | > 5% |
| 3 | Latencia p95 | dato en posición 95% | < 100 ms | > 500 ms |
| 4 | Rate por Status | peticiones por código / segundos | Mayoría 200 | Pico de 401/500 |
| 5 | p50 / p95 / p99 | posición 50% / 95% / 99% | p99 ≈ p95 | p99 >> p95 |
| 6 | Heap Memory | (usada / total) × 100 | < 70% | Sube sin bajar |
| 7 | CPU Usage | (CPU s / tiempo s) × 100 | < 30% | > 70% sostenido |
| 8 | Event Loop Lag | tiempo real − tiempo esperado | < 10 ms | > 100 ms |
| 9 | Active Handles | conteo directo | Proporcional al tráfico | Sube sin bajar |

---
### Ejercicio final

Dado este snapshot del servidor, calcula los 4 KPIs que puedas:

| Dato | Valor |
|---|---|
| Peticiones totales en 5 min | 800 |
| Peticiones con error 500 | 40 |
| Segundos observados | 300 |
| Heap usada | 130 MB |
| Heap total | 150 MB |
| Segundos de CPU usados | 12 |
| Segundos reales | 60 |

In [ ]:
peticiones_totales = 800
peticiones_con_error = 40
segundos = 300
heap_usada_mb = 130
heap_total_mb = 150
segundos_cpu = 12
segundos_reales = 60

request_rate = peticiones_totales / segundos
error_rate = (peticiones_con_error / peticiones_totales) * 100
heap_uso = (heap_usada_mb / heap_total_mb) * 100
cpu_uso = (segundos_cpu / segundos_reales) * 100

print("request_rate =", request_rate)
print("error_rate =", error_rate)
print("heap_uso =", heap_uso)
print("cpu_uso =", cpu_uso)